# Multimodal Laughter Detection - Text + Audio Fusion

## Goal: F1 ≥ 0.75 per language (7 languages)

### Architecture
```
Text Branch:  XLM-RoBERTa-base (768-dim) → text_emb
Audio Branch: Wav2Vec2-base (768-dim) → audio_emb
Fusion:       CrossAttention([text_emb, audio_emb]) → classifier
```

### Dataset
| Modality | Language | Examples |
|----------|----------|----------|
| Text | EN, ZH, HI-LATN, FR, ES, BN | 19,109 |
| Audio | EN (subset with transcripts) | ~5 hours |

### Target Metrics
- Overall F1 ≥ 0.75
- Per-language F1 ≥ 0.70 (min 5 examples)
- Audio-only baseline F1 ≥ 0.65 (when data available)


In [ ]:
!pip install -q torch transformers scikit-learn pandas numpy torchaudio

In [ ]:
import torch
import pandas as pd
import numpy as np
import json
import os
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, Wav2Vec2Model, Wav2Vec2Processor
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

TEXT_DIR = '/content/drive/MyDrive/merged_final'
AUDIO_DIR = '/content/drive/MyDrive/audio_comedy'
MODEL_DIR = '/content/drive/MyDrive/multimodal_models'
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Text: {TEXT_DIR}')
print(f'Audio: {AUDIO_DIR}')
print(f'Output: {MODEL_DIR}')

In [ ]:
# Load text dataset (already processed)
def load_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

print('Loading text dataset...')
train_df = load_jsonl(f'{TEXT_DIR}/train.jsonl')
val_df = load_jsonl(f'{TEXT_DIR}/valid.jsonl')
print(f'Train: {len(train_df)}, Val: {len(val_df)}')
print('\nLanguages:', train_df['language'].value_counts().to_dict())

In [ ]:
# Prepare binary labels from word-level annotations
def prepare_labels(df):
    if 'labels' in df.columns and isinstance(df.iloc[0].get('labels', []), list):
        df['ex_label'] = df['labels'].apply(lambda x: 1 if any(v == 1 for v in x) else 0)
    elif 'label' in df.columns:
        df['ex_label'] = df['label']
    return df

train_df = prepare_labels(train_df)
val_df = prepare_labels(val_df)
print(f'Train label dist: {train_df["ex_label"].value_counts().to_dict()}')

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tok, max_len=128):
        self.texts, self.labels, self.tok, self.max_len = texts, labels, tok, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        txt = ' '.join(self.texts[idx]) if isinstance(self.texts[idx], list) else str(self.texts[idx])
        enc = self.tok(txt, add_special_tokens=True, max_length=self.max_len, 
                        padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(), 
                'attention_mask': enc['attention_mask'].flatten(),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

print('Preparing text data...')
train_texts = train_df['words'].apply(lambda x: ' '.join(x)).tolist()
train_labels = train_df['ex_label'].tolist()
val_texts = val_df['words'].apply(lambda x: ' '.join(x)).tolist()
val_labels = val_df['ex_label'].tolist()

text_tok = AutoTokenizer.from_pretrained('xlm-roberta-base')
train_ds = TextDataset(train_texts, train_labels, text_tok)
val_ds = TextDataset(val_texts, val_labels, text_tok)
train_ld = DataLoader(train_ds, batch_size=16, shuffle=True)
val_ld = DataLoader(val_ds, batch_size=16)
print(f'Text batches: train={len(train_ld)}, val={len(val_ld)}')

In [ ]:
class TextEncoder(torch.nn.Module):
    """XLM-RoBERTa text encoder → 768-dim embedding"""
    def __init__(self):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained('xlm-roberta-base')
        self.proj = torch.nn.Linear(768, 256)  # Project to 256-dim
    def forward(self, ids, mask):
        out = self.xlmr(input_ids=ids, attention_mask=mask)
        cls = out.last_hidden_state[:, 0]  # [CLS] token
        return self.proj(cls)  # 256-dim text embedding

text_enc = TextEncoder().to(DEVICE)
print('Text encoder ready!')

In [ ]:
class AudioEncoder(torch.nn.Module):
    """Wav2Vec2 audio encoder → 768-dim → 256-dim embedding"""
    def __init__(self):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')
        self.proj = torch.nn.Linear(768, 256)  # Project to 256-dim
        # Freeze wav2vec2 initially (train projection first)
        for p in self.wav2vec.parameters():
            p.requires_grad = False
    def forward(self, wav):
        out = self.wav2vec(wav)
        # Use mean pooling of last hidden state
        emb = out.last_hidden_state.mean(dim=1)
        return self.proj(emb)  # 256-dim audio embedding

audio_enc = AudioEncoder().to(DEVICE)
print('Audio encoder ready!')

In [ ]:
class MultimodalFusion(torch.nn.Module):
    """
    Cross-attention fusion of text and audio embeddings
    
    Architecture:
        text_emb: [B, 256]
        audio_emb: [B, 256]
        concat → cross-attn → classifier
    """
    def __init__(self, embed_dim=256):
        super().__init__()
        # Cross-attention: text attends to audio and vice versa
        self.cross_attn = torch.nn.MultiheadAttention(embed_dim, heads=4, batch_first=True)
        self.norm = torch.nn.LayerNorm(embed_dim)
        
        # Classifier head
        self.fc1 = torch.nn.Linear(embed_dim * 2, 128)
        self.drop = torch.nn.Dropout(0.2)
        self.fc2 = torch.nn.Linear(128, 2)
        
    def forward(self, text_emb, audio_emb):
        # Reshape for cross-attn: [B, 1, 256]
        t = text_emb.unsqueeze(1)
        a = audio_emb.unsqueeze(1)
        
        # Cross-attention: text attends to audio
        attn_out, _ = self.cross_attn(t, a, a)
        fused = self.norm(attn_out.squeeze(1))
        
        # Concatenate original embeddings
        concat = torch.cat([text_emb, audio_emb], dim=1)
        
        # Final classifier
        x = self.drop(torch.nn.functional.relu(self.fc1(concat)))
        return self.fc2(x)

class MultimodalLaughDetector(torch.nn.Module):
    """Full multimodal model: text encoder + audio encoder + fusion"""
    def __init__(self):
        super().__init__()
        self.text_enc = TextEncoder()
        self.audio_enc = AudioEncoder()
        self.fusion = MultimodalFusion()
        
    def forward(self, text_ids, text_mask, audio_wav):
        text_emb = self.text_enc(text_ids, text_mask)
        audio_emb = self.audio_enc(audio_wav)
        return self.fusion(text_emb, audio_emb)
    
    def text_only(self, text_ids, text_mask):
        """Text-only inference mode"""
        text_emb = self.text_enc(text_ids, text_mask)
        # Use a placeholder audio embedding of zeros
        audio_emb = torch.zeros_like(text_emb)
        return self.fusion(text_emb, audio_emb)

print('Multimodal model architecture ready!')

In [ ]:
def train_text_only(model, train_ld, val_ld, epochs=10, lr=2e-5):
    """Train text-only (use audio_emb=zeros when audio unavailable)"""
    opt = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    criterion = torch.nn.CrossEntropyLoss()
    
    best_f1 = 0
    for ep in range(epochs):
        # Train
        model.train()
        train_loss = 0
        preds, labs = [], []
        for b in train_ld:
            opt.zero_grad()
            logits = model(b['input_ids'].to(DEVICE), 
                          b['attention_mask'].to(DEVICE), 
                          torch.zeros(1).to(DEVICE))  # dummy audio
            loss = criterion(logits, b['label'].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss += loss.item()
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labs.extend(b['label'].numpy())
        train_f1 = f1_score(labs, preds, average='weighted')
        
        # Val
        model.eval()
        val_preds, val_labs = [], []
        with torch.no_grad():
            for b in val_ld:
                logits = model(b['input_ids'].to(DEVICE),
                              b['attention_mask'].to(DEVICE),
                              torch.zeros(1).to(DEVICE))
                val_preds.extend(torch.argmax(logits, 1).cpu().numpy())
                val_labs.extend(b['label'].numpy())
        val_f1 = f1_score(val_labs, val_preds, average='weighted')
        
        print(f'Ep{ep+1}: TL={train_loss/len(train_ld):.4f} TF={train_f1:.4f} VF={val_f1:.4f}')
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f'{MODEL_DIR}/multimodal_best.pt')
            print(f'  → Saved F1={best_f1:.4f}')
    
    return best_f1

print('Training function ready!')

In [ ]:
# Initialize multimodal model
model = MultimodalLaughDetector().to(DEVICE)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

# Train text-only (audio data not ready for full multimodal)
# Audio branch will be trained when we have labeled audio data
print('\nTraining text-only mode (audio branch will be trained later)...')
best_f1 = train_text_only(model, train_ld, val_ld, epochs=10)

In [ ]:
# Load best model and evaluate per language
model.load_state_dict(torch.load(f'{MODEL_DIR}/multimodal_best.pt'))
model.eval()

val_preds = []
with torch.no_grad():
    for b in val_ld:
        logits = model(b['input_ids'].to(DEVICE),
                      b['attention_mask'].to(DEVICE),
                      torch.zeros(1).to(DEVICE))
        val_preds.extend(torch.argmax(logits, 1).cpu().numpy())

val_df['pred'] = val_preds

print('='*50)
print('PER-LANGUAGE RESULTS')
print('='*50)
results = {}
for lang in sorted(val_df['language'].unique()):
    ld = val_df[val_df['language'] == lang]
    if len(ld) > 5:
        f1 = f1_score(ld['ex_label'], ld['pred'], average='weighted')
        results[lang] = f1
        st = 'PASS' if f1 >= 0.70 else 'FAIL'
        print(f'{lang.upper()}: F1={f1:.4f} [{st}] (n={len(ld)})')

overall = f1_score(val_df['ex_label'], val_df['pred'], average='weighted')
print(f'\nOVERALL: F1={overall:.4f}')
print(f'Best Val F1: {best_f1:.4f}')

In [ ]:
# Save results
with open(f'{MODEL_DIR}/results.json', 'w') as f:
    json.dump({
        'best_val_f1': best_f1,
        'overall_f1': overall,
        'per_language': results,
        'mode': 'text_only (audio branch not trained)'
    }, f, indent=2)

print(f'\nResults saved to {MODEL_DIR}/results.json')
print(f'Model saved to {MODEL_DIR}/multimodal_best.pt')

## Next Steps: Audio Training

Once we have **labeled audio data** (laughs timestamped in audio):

### 1. Collect Audio Data
```bash
python training/audio_data_collector.py --comedian "Zakir Khan" --max_videos 3
python training/audio_data_collector.py --comedian "Dave Chappelle" --max_videos 3
```

### 2. Add Laugh Annotations
Use text laughter detection to pseudo-label audio regions:
```python
# Run text model on transcript words
# Map laugh predictions to audio timestamps
# Create audio manifest with laugh labels
```

### 3. Train Audio Branch
```python
# Unfreeze wav2vec2 and train on labeled audio
audio_enc.unfreeze_wav2vec()
# Train audio branch alone
# Then train full multimodal fusion
```

### Target
- Audio-only F1 ≥ 0.65
- Full multimodal F1 ≥ 0.78
